# Retail order cleansing and customer profiling

Rebuild dirty, missing and retained-outlier outputs, reconcile every change with a field-level audit, and compare customer-feature transformations.

This is a four-person collaborative pipeline. Detailed roles are recorded in the root README. Private input records and saved record-level outputs are intentionally not included.

## Configuration

Run from this folder after preparing a verified local package in `data/` using the root preparation utility. Set `RETAIL_DATA_DIR` and `RETAIL_OUTPUT_DIR` to override local paths. Processing is offline and the VADER resource is verified against its local contract.

In [ ]:
from pathlib import Path
import os, sys, json, importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

DATASET_ID = "Retail"
DATA_DIR = Path(os.environ.get("RETAIL_DATA_DIR", "./data"))
OUTPUT_DIR = Path(os.environ.get("RETAIL_OUTPUT_DIR", "./outputs"))
KNOWN_INPUT_LIMITATIONS = []

from pipeline import run_dirty, run_missing, run_outlier, initial_investigation
from validation import verify_input_manifest, reconcile_outputs

def show_table(title, frame, precision=4):
    display(Markdown(f"**{title}**"))
    with pd.option_context("display.max_colwidth", None):
        table = frame.to_html(border=0, classes="Retail-table", na_rep="Undefined",
                              float_format=lambda value: f"{value:.{precision}f}", escape=True)
    css = "<style>.Retail-table td,.Retail-table th{padding:7px;vertical-align:top;text-align:left;max-width:320px;white-space:normal;} .Retail-table thead{background:#edf3f5;}</style>"
    display(HTML(css + '<div style="overflow-x:auto">' + table + '</div>'))

print("Python", sys.version.split()[0])
show_table("Runtime packages", pd.DataFrame([
    {"package": name, "version": metadata.version(name)}
    for name in ["numpy", "pandas", "matplotlib", "nltk", "scikit-learn"]
]))
input_manifest_before = verify_input_manifest(DATA_DIR, DATASET_ID)
show_table("Allocated input checksums", pd.DataFrame(input_manifest_before))

## Cleansing — Data cleansing and explanation
### 1.0 Initial investigation and reliable evidence

The batches contain distinct orders. Read raw fields as text to retain the
original representation and identify empty CSV cells. Convert numeric fields
only for their calculations. Dirty rows contain at most one incorrect field;
the missing batch's observed values are valid; the outlier batch has anomalous
delivery charges only. These guarantees guide evidence selection, not a fixed
percentage of records or patterns in IDs.

Outlier-batch cart equations are reliable evidence for current product prices.
The missing batch supplies complete observed model fields for seasonal delivery
models. Every fitted relationship is documented below. A failure of a total
check alone does not identify which field is wrong.

In [ ]:
eda = initial_investigation(DATA_DIR, DATASET_ID)
for name, table in eda.items():
    show_table(name, table)

### 1.1 Dirty data — field attribution and preservation

Dirty's rule-based component is rerun from the original package. Its one-field
repairs use independent evidence: current cart prices, total arithmetic,
calendar/season consistency, nearest-warehouse Haversine distance, the exact
frozen VADER rule, and seasonal delivery residuals. Correctly observed linked
fields are retained rather than repeatedly changed to force agreement.

Dates are parsed as calendar dates with day-first slash dates and written as
ISO dates. Haversine uses Earth radius 6,371 km. Coordinate precision is retained.
VADER uses the supplied lexicon checksum and NLTK version; the exact no-review
sentinel implies happy, otherwise compound >= 0.05 implies happy. Product prices
are inferred from current cleansing equations rather than historical integration prices.

The component's evidence file retains detailed per-order diagnoses, model
metrics and checks. The generated audit points to this evidence. The shared
price-inference method is also independently checked on held-out cart equations
in section 1.3.

In [ ]:
dirty_result = run_dirty(DATA_DIR, OUTPUT_DIR)
dirty_audit = pd.read_csv(OUTPUT_DIR / "audit_log_dirty.csv", dtype=str, keep_default_na=False)
show_table("Dirty repairs by field", dirty_audit.groupby("field_name").size().rename("repaired_cells").to_frame())
show_table("Examples of traceable dirty-data decisions", dirty_audit.head(12))
display(Markdown(dirty_result["summary"]))
for title, table in dirty_result["tables"].items():
    show_table(title, table)

### 1.2 Missing data — field-appropriate recovery

The source has missing fields only. Calendar, warehouse and distance rules,
frozen VADER, inferred current prices and the required total formula recover
calculable values. Each cart item amount is rounded to cents before summing;
the coupon discounts items only, then delivery is added before final rounding.
No GST or integration-derived fields are added.

Missing trains seasonal regressions on complete observed model fields using
an 80/20 split with `train_test_split(random_state=5196)`, reports held-out
performance, then refits on all eligible observations for imputation. Missing
expedited flags are chosen by comparing the False and True delivery residuals.
Missing delivery charges are predicted and rounded to cents. Observed totals
also permit an independent algebraic check of those charge imputations.

This integration invokes the delivered recovery functions through a local-folder
adapter. The adapter also reruns the component's validation on exported files.
Observed values are preserved; each imputation contributes one audit entry.

In [ ]:
missing_result = run_missing(DATA_DIR, OUTPUT_DIR)
missing_counts = pd.Series(missing_result["evidence"]["missing_counts_before"])
show_table("Missing fields before recovery", missing_counts[missing_counts.gt(0)].rename("missing_cells").to_frame())
show_table("Missing-component price inference", pd.Series(missing_result["evidence"]["price_inference"]["prices"], name="current_unit_price_AUD").to_frame())
show_table("Missing-component held-out seasonal performance", pd.DataFrame(missing_result["evidence"]["delivery_model_evaluation"]).T)
show_table("Missing-component read-back validation", pd.DataFrame(missing_result["checks"]))
display(Markdown(f"Independent observed-total check: **{missing_result['independent_charge_check']['rows']}** imputed charges; maximum difference **AUD {missing_result['independent_charge_check']['max_abs_difference_aud']:.2f}**."))

### 1.3 Outlier data — seasonal residuals and independent corroboration

Outlier's delivered Notebook calculations are wrapped as a callable component,
with paths made configurable. The model still uses an intercept, distance,
expedited 0/1 and happy 0/1 separately for each season. Its seeded NumPy split
differs from Missing's scikit-learn split, and its retained training-fit
coefficients differ slightly from Missing's all-observation refit. These are
separate documented fits; their coefficients are not claimed to be identical.

The residual rule removes rows with absolute observed-minus-predicted delivery
charge above AUD 0.02. This is a diagnostic choice justified by held-out errors,
residual separation and threshold sensitivity, not the export tolerance.
Retained rows preserve their original values. Independent total arithmetic,
raw-charge screens and a retained-row refit corroborate the decisions.

In [ ]:
outlier_result = run_outlier(DATA_DIR, OUTPUT_DIR, DATASET_ID)
show_table("Outlier batch: initial field investigation", outlier_result["outlier_eda"])
show_table("Outlier model: held-out performance", outlier_result["evaluation"], 9)
show_table("Outlier model: fitted coefficients", outlier_result["coefficients"], 6)
show_table("Dirty / Outlier diagnostic model agreement", outlier_result["dirty_model_alignment"], 12)
print("Dirty / Missing price agreement:", outlier_result["current_price_alignment"])
show_table("Residual decision and separation", outlier_result["decisions"], 6)
show_table("Threshold sensitivity", outlier_result["thresholds"])
show_table("Independent arithmetic corroboration", outlier_result["corroboration"])
show_table("Retained-row coefficient refit", outlier_result["refit_comparison"], 6)
print("Maximum held-out current-price equation error (AUD):", outlier_result["price_holdout_errors"].max())
for figure in outlier_result["figures"]:
    display(figure)
# Include Outlier's raw-charge-screen comparison table.
for evidence in outlier_result["evidence"]:
    value = evidence["value"]
    if isinstance(value, pd.DataFrame) and "screen" in value.columns:
        show_table("Alternative raw-charge screens", value)

### 1.4 Export boundaries

All three components have now exported the required cleaned files with the
original 16-column schema, column order and units. Dirty and missing retain
all 500 original order IDs each; outlier retains only the unflagged orders.
No analysis flags or pandas index are exported. The following checks reread
these files and reconcile actual changes with the generated combined audit.

## Audit validation — Audit trail and validation

The audit uses one row per changed field and one per removed outlier. Missing
source cells use `<MISSING>`; removed rows use `delivery_charges` and
`ROW_REMOVED`. Unchanged values and Customer profiling transformations are excluded.

Confidence labels are qualitative: **high** means a published rule or strongly
corroborated decision; **medium** means a validated model is needed for
attribution/estimation; **low** means insufficient evidence for a unique repair,
which must remain explicitly unresolved rather than fabricated.

Outlier's checks include schema, IDs, completeness, business rules, retained
residuals and audit reconciliation. Integration adds independent cell-by-cell
reconciliation using exact decimal numeric equivalence, booleans and cart
semantics. No saved PASS output is taken as evidence without rerunning it.

In [ ]:
show_table("Cross-file read-back and audit checks", outlier_result["cross_file_validation"])
show_table("Business-rule failure counts by output", outlier_result["business_failures"])
integration_reconciliation = reconcile_outputs(DATA_DIR, OUTPUT_DIR, DATASET_ID)
show_table("Independent actual-change / audit reconciliation", integration_reconciliation)
show_table("Combined audit counts", outlier_result["audit"].groupby(["source_file", "issue_type"]).size().rename("entries").to_frame())
assert integration_reconciliation["passed"].all()
assert outlier_result["cross_file_validation"]["passed"].all()
assert not outlier_result["business_failures"].to_numpy().any()

## Customer profiling — Customer profiling and transformation

Prepare activity, spending and delivery-geography attributes for possible future
distance-based exploratory segmentation. The next cells reread the final three
cleaned outputs produced above. Profiles and transformed values remain in this
Notebook only. There is no clustering, prediction or dashboard requirement.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.metadata as metadata

# DATASET_ID is configured in section 0.
CLEAN_DIR = OUTPUT_DIR
RAW_DIR = DATA_DIR
# KNOWN_INPUT_LIMITATIONS is configured in section 0.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
from profiles import (
    FEATURES, METHODS, input_status, load_cleaned_orders, build_profiles,
    describe_features, compare_transformations, distance_effects,
    recommendation_table, interpretation, analysis_checks, make_figures,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 200)
print("Python:", sys.version.split()[0])
print({name: metadata.version(name) for name in ("numpy", "pandas", "matplotlib", "ipykernel")})
print("Clean input directory:", CLEAN_DIR.resolve())

def show_table(title, frame, precision=4):
    display(Markdown(f"**{title}**"))
    with pd.option_context("display.max_colwidth", None):
        table = frame.to_html(border=0, classes="profiles-table", na_rep="Undefined",
                              float_format=lambda value: f"{value:.{precision}f}", escape=True)
    css = "<style>.profiles-table th,.profiles-table td {padding:8px;vertical-align:top;text-align:left;max-width:310px;white-space:normal;} .profiles-table thead {background:#edf3f5;}</style>"
    display(HTML(css + '<div style="overflow-x:auto">' + table + '</div>'))

def pending():
    display(Markdown("**Pending:** provide the three final cleaned CSVs and rerun all cells. No sample results are substituted."))

## 3.1 Build the customer-level analysis copy

Read the cleaned dirty, missing and retained-outlier orders, preserving
`source_batch` and `source_file` in the analysis copy. The three files contain
different orders; concatenate them rather than joining on `order_id`. Reject
duplicates rather than silently dropping them. Dirty and missing must retain
500 orders each; outlier may contain only the retained original orders.

We select exactly three specified attributes:

| Attribute | Definition over retained orders | Role and unit |
|---|---|---|
| `order_count` | Distinct `order_id` count per customer | Activity, orders |
| `mean_order_value` | Mean `order_total` per customer | Spending per order, AUD |
| `mean_delivery_distance` | Mean `distance_to_nearest_warehouse` | Delivery geography, km |

`order_total` includes delivery after the item discount. It is not profit or
merchandise-only revenue. We omit `total_paid`, avoiding the exact identity
total = count × mean across a count/total/mean selection. The chosen attributes
can still be correlated, which we inspect below. `customer_id` is an unchanged
identifier and is never scaled or included in a distance.

These are retained cleansing sample profiles. Removing an order can change a customer's
count and means, and can remove a customer whose only orders were outliers.
Do not restore rows to reach 1,500, append integration orders or use another group's data.

In [ ]:
status = input_status(CLEAN_DIR, DATASET_ID)
show_table("Required cleaned inputs", status)
READY = bool(status["exists"].all())
if READY:
    orders, batch_counts, input_checks, input_fingerprints = load_cleaned_orders(
        CLEAN_DIR, DATASET_ID, RAW_DIR
    )
    profiles = build_profiles(orders)
    baseline = profiles.copy(deep=True)
    show_table("T3-A. Retained input sample by batch", batch_counts)
    print(f"Combined distinct orders: {orders.order_id.nunique():,}")
    print(f"Distinct customers / profile rows: {len(profiles):,}")
    print(f"Removed outlier orders: {500 - int(batch_counts.loc[batch_counts.source_batch.eq('outlier'), 'retained_rows'].iloc[0]):,}")
    print("Batch customer counts overlap and must not be added together.")
    show_table("Input prerequisites (business-rule validation remains in Audit validation)", input_checks)
    show_table("Customer profiles — first 10 rows; full table remains in profiles", profiles.head(10))
    if KNOWN_INPUT_LIMITATIONS:
        display(Markdown("**Declared upstream limitations:**\n\n" + "\n".join(f"- {x}" for x in KNOWN_INPUT_LIMITATIONS)))
    else:
        print("No upstream limitations declared here. Refer to the group's Audit validation evidence.")
else:
    pending()

## 3.2 Original scales, distributions and relationships

Keep an untouched baseline. Table T3-B reports minimum, quartiles, median, mean,
95th percentile, maximum, population standard deviation (`ddof=0`), sample
skewness, zero counts and constant-feature flags. Undefined skewness for a
constant or very small sample is shown explicitly. Figure 1 shows distributions
and pairwise relationships; Table T3-C quantifies linear association.

Euclidean distance squares numeric differences, so AUD, kilometres and counts
are not directly comparable. A large spread can dominate distance even when
that attribute is no more important to the business. Low skewness does not
remove this scale problem. Extreme values in this cleaned sample may be valid;
Customer profiling does not delete them.

In [ ]:
if READY:
    baseline_summary = describe_features(baseline)
    show_table("T3-B. Original distributions and scales", baseline_summary)
    show_table("T3-C. Original Pearson correlations", baseline.corr())
else:
    pending()

## 3.3 Compare z-score standardisation and min-max scaling

Apply both methods separately to **all three** attributes:

\[
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j},\qquad
m_{ij}=\frac{x_{ij}-\min(x_j)}{\max(x_j)-\min(x_j)}.
\]

Population SD uses `ddof=0`. Every nonconstant z-score feature has mean zero
and variance one; it is not restricted to [0, 1]. Min-max puts the observed
minimum and maximum at 0 and 1, but does not equalise feature variances.
Both are positive affine transformations: they preserve within-feature order
and Pearson correlations (where defined), but change between-feature weighting.

For a constant attribute, both implementations return zero and mark the
attribute as non-informative for distance. No division by zero, clipping or
rounding of analytical values is used. Min-max is sensitive to the observed
endpoints; standardisation is also sensitive to extremes through mean and SD.
Scaling parameters are retained for inspection. They are fitted to this entire
descriptive sample; any later held-out model must fit its preprocessing on its
training data only.

In [ ]:
if READY:
    comparisons = compare_transformations(baseline)
    transformed_features = comparisons["versions"]
    effects = distance_effects(transformed_features)
    figures = make_figures(baseline, comparisons, effects)
    display(figures["baseline"])
    show_table("T3-D1. Fitted scaling parameters", comparisons["parameters"])
    show_table("T3-D2. Original versus z-score versus min-max", comparisons["scaling_summary"])
    display(figures["scaling"])
    # Figures are displayed explicitly; close them to avoid duplicate inline output.
    for figure in figures.values():
        plt.close(figure)
else:
    pending()

## 3.4 Compare a nonlinear alternative and its distance effects

Compare mean order value with `log1p(value)` using a fixed reference of 1 AUD:
\(\log(1+x/1\text{ AUD})\). Mathematically the numerical input must exceed -1;
our monetary input is required to be non-negative. Zero maps to zero. Ordinary
log would require strictly positive values. No arbitrary offset is estimated.

The transformation is increasing and concave: customer order by spending is
preserved, while high-value gaps are compressed. It changes the interpretation
of a unit difference, so less skewness alone is not sufficient to recommend it.
Table T3-E and Figure 3 show the observed effect rather than assuming success.

The combined alternative applies **log1p to mean order value first**, then fits
z-score scaling to all three attributes. Count and geography receive no nonlinear
transformation. Table T3-F and Figure 4 compare the four whole-profile
representations using exact distances over all customer pairs.

For any nonconstant representation, the total squared-distance contribution of
feature \(j\) across all pairs equals its variance divided by the sum of feature
variances. This exposes implicit weighting. The pairwise-distance rank
correlation with the original representation uses average ranks for ties.
Undefined correlations or shares are retained when there is insufficient
variation. A lower distance correlation means a changed geometry, not better
segmentation. These calculations do not fit a predictive or clustering model.

In [ ]:
if READY:
    show_table("T3-E. Mean order value before and after log1p", comparisons["nonlinear_summary"])
    display(figures["nonlinear"])
    show_table("T3-F. Relative distances and feature contributions", effects)
    display(figures["distance_shares"])
    show_table("T3-G. Pearson relationships after log1p value then z-score",
               comparisons["correlations"]["Log1p value + Z-score"])
    for observation in interpretation(baseline, comparisons, effects):
        display(Markdown(observation))
else:
    pending()

## 3.5 Recommendations, boundaries and limitations

The table below applies an explicit, interpretable starting policy: compare
customers using equal marginal feature variance and preserve absolute
within-feature increments. For nonconstant features this recommends **no
nonlinear transformation; z-score standardisation**. A constant feature is
excluded from the distance calculation (its scaled value is zero).

This is one defensible policy, not a data-optimised universal answer. The table
reports the observed ranges and distance contributions from this run. For
spending it also reports the log comparison. We retain linear spending because
this interpretation values absolute AUD gaps; if the business instead values
relative spending differences, the measured log1p-then-z-score alternative is
available. Review the actual figures and revise the recommendation if that
stated purpose does not match the group's intended comparison.

The three selected attributes avoid the count/total/mean identity, but may still
be redundant. Equal variance is an explicit weighting choice, not equal business
importance. Valid extremes may remain influential. Counts depend on the observed
sample window; means based on one order are less stable than means based on many.
Outlier removal and upstream estimation can affect profiles. Retained customers
are not a complete population or lifetime history. No comparison establishes
meaningful clusters. Future predictive validation requires training-only
preprocessing; no held-out model is required for this task.

In [ ]:
if READY:
    recommendations = recommendation_table(baseline, comparisons, effects)
    show_table("T3-H. Per-attribute recommendations and evidence", recommendations)
else:
    pending()

## Reproducibility and preservation checks

Reconcile customer counts and weighted means to retained orders. Check finite
transformations, scaling invariants, unchanged identifiers, inverse log1p and
the preserved original baseline. Compare SHA-256 values of the input CSVs and,
when present, `audit_log.csv` before and after the calculation. These checks
validate Customer profiling calculations; they do not certify the upstream cleaning methods.

Keep customer profiles and transformed features in the notebook only. Do not
replace the values in the three exported CSVs or add Customer profiling transformations to
the cleaning audit log. After integration, Restart Kernel and Run All from the
original inputs through Tasks 1–3, keeping useful outputs visible.

In [ ]:
if READY:
    pd.testing.assert_frame_equal(profiles, baseline)
    checks = analysis_checks(orders, profiles, comparisons, input_fingerprints)
    show_table("T3-I. Executable Customer profiling checks", checks)
    print(f"Customer profiling executed on {len(orders):,} retained orders and {len(profiles):,} customers.")
    print("Review T3-H with the plots and upstream Audit validation findings before release.")
else:
    display(Markdown("**Customer profiling result status: NOT YET GENERATED.** The code is ready, but final cleaned inputs are required. This notebook is not a completed cleansing release."))

## Reproducibility and preservation

The next cell verifies input checksums, complete customer-profile execution and actual-change/audit reconciliation. Local outputs may contain sensitive records and must not be committed.

In [ ]:
input_manifest_after = verify_input_manifest(DATA_DIR, DATASET_ID)
assert input_manifest_before == input_manifest_after, "Original inputs were modified"
assert READY, "Customer profiling requires all three final cleaned outputs"
final_summary = {
    "dataset_id": DATASET_ID,
    "raw_input_checksums_unchanged": True,
    "retained_orders": int(len(orders)),
    "customers": int(len(profiles)),
    "audit_entries": int(len(outlier_result["audit"])),
    "reconciliation": integration_reconciliation.to_dict("records"),
    "profiles_checks_passed": bool(checks.result.eq("PASS").all()),
    "profiles_check_count": int(len(checks)),
    "dirty_outlier_model_alignment": outlier_result["dirty_model_alignment"].to_dict("records"),
    "dirty_missing_price_alignment": outlier_result["current_price_alignment"],
}
(OUTPUT_DIR / "integration_validation.json").write_text(json.dumps(final_summary, indent=2), encoding="utf-8")
display(Markdown(f"**Run complete:** {len(orders):,} retained orders, {len(profiles):,} customers, {len(outlier_result['audit']):,} audit entries. Original input checksums are unchanged."))